# Ensemble Methods: Random Forests and Gradient Boosting

**Companion notebook to Chapter 54 — Ensemble Methods: Random Forests and Gradient Boosting.**

*Build one good model, or build many imperfect ones and let them vote?* Leo Breiman's 2001 answer — the **random forest** — showed that the aggregate of many weak learners is often more trustworthy than any single polished model, and that insight now anchors most practical machine learning on tabular engineering data.

An **ensemble method** combines many simpler models — each individually imperfect — into a combined prediction that is more accurate and more robust than any single component. The intuition is that different models make different errors: averaging their predictions cancels out individual mistakes, leaving the signal that all of them agree on. **Random forests** and **gradient boosting** are the two most successful ensemble approaches in practice, and both are built from decision trees.

A random forest grows trees *in parallel*, each on a bootstrap sample of the training data and a random subset of features. Because the trees are independent and their predictions are averaged, the ensemble resists overfitting even with very deep individual trees. Gradient boosting grows trees *sequentially*: each new tree is trained specifically to predict the residual error of the ensemble built so far. Gradient boosting typically achieves higher accuracy on structured tabular data; random forests are more forgiving of hyperparameter choices and less sensitive to outliers.

Both methods provide an output that neural networks and linear models typically do not: *feature importance* — a quantitative ranking of which input features most influenced the classification. For structural health monitoring this is directly actionable: features with low importance represent sensors that do not contribute to the diagnosis and could potentially be removed without degrading performance, while features with high importance identify the measurements most critical to monitor continuously. As the chapter stresses, an importance score is a *physical hypothesis to be tested*, not proof of causation.

**This notebook demonstrates:**
1. **Three-class damage-state classification** (healthy, minor crack, major damage) with a random forest and a gradient-boosting classifier — comparing their classification reports and their feature-importance rankings (Figure 54.2).
2. **A simulated bridge-monitoring example** with twenty gages, where pairwise sensor correlations — not individual peak strains — emerge as the top predictors of damage state (Figure 54.3).

In [ ]:
# Colab setup: install dependencies (uncomment to run in Google Colab)
# !pip install numpy matplotlib scikit-learn

import pathlib

import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, accuracy_score

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11, 'axes.titlesize': 12})

# Reproducibility: one shared seed for every sklearn estimator and data split.
RANDOM_STATE = 0

# All notebook figures are written here. These paths are referenced verbatim by
# the book, so do not rename them.
OUT_DIR = pathlib.Path('../src/media/ch54')
OUT_DIR.mkdir(parents=True, exist_ok=True)

---
## 1 — Three-Class Damage Classification: Random Forest vs. Gradient Boosting

The dataset simulates 600 specimens per class, drawn from multivariate normal distributions representing three structural states: *healthy* (baseline vibration level), *minor crack* (slightly elevated peak strain and spectral centroid), and *major damage* (substantially elevated on all four features). The four features — **peak strain**, **RMS strain**, **spectral centroid frequency**, and **high-amplitude cycle count** — stand in for what a real monitoring system would compute from continuous strain recordings. Every draw is seeded, so the dataset is identical on each run.

We fit both ensembles on the same 80/20 *stratified* split and read off two things. The **classification reports** quantify precision, recall, and F1 for each class and each model — reported *per class* rather than collapsed into a single accuracy, because (as the chapter argues) one headline accuracy hides rare-class performance. Both models should score well here given the clear class separation; the more revealing comparison is the **feature-importance** chart.

The importance bar chart (Figure 54.2) places both models side by side. The spectral centroid is the most cleanly separated feature and the cycle count the next, so we expect those two to dominate the ranking with peak and RMS strain secondary — and both algorithms should agree on that order. Agreement between two differently-built models is a practical sign that the ranking reflects the *data* rather than an artifact of any one algorithm. One caution the chapter emphasizes: the two impurity-based importances are each normalized *within* their own model, so compare their **rankings**, not their raw magnitudes. This is exactly the output a monitoring engineer would use to rationalize sensor placement — a channel with near-zero importance in both models is a candidate to drop from a lower-cost system.

In [ ]:
# --- Hyperparameters and dataset design (named, not magic numbers) --------------
N_PER_CLASS   = 600     # specimens simulated per damage state
TEST_SIZE     = 0.2     # fraction held out for evaluation
N_ESTIMATORS  = 200     # trees in each ensemble
LEARNING_RATE = 0.05    # gradient-boosting shrinkage (eta, Eq. 54.1)
DATA_SEED     = 42      # seeds the synthetic feature draws

# Per-class feature means: [peak strain (µε), RMS strain (µε),
#                           spectral centroid (kHz), high-amplitude cycle count]
CLASS_MEANS = {
    'Healthy':      [600, 280, 0.42, 45],
    'Minor crack':  [720, 340, 0.51, 62],
    'Major damage': [910, 430, 0.63, 88],
}
# Shared diagonal covariance (features treated as independent within a class).
FEATURE_COV = np.diag([120**2, 80**2, 0.05**2, 15**2])

FEATURE_LABELS = ['Peak strain\n(µε)', 'RMS strain\n(µε)',
                  'Spectral\ncentroid (kHz)', 'Cycle count\n(above threshold)']


def make_damage_dataset(n_per_class=N_PER_CLASS, seed=DATA_SEED):
    """Synthesize a labelled tabular strain-feature dataset.

    Draws ``n_per_class`` samples for each of the three damage states from a
    multivariate normal with per-class means (``CLASS_MEANS``) and a shared
    diagonal covariance (``FEATURE_COV``). Seeding makes the draw reproducible.

    Returns
    -------
    X : ndarray, shape (3 * n_per_class, 4)
        Feature matrix, one row per simulated specimen.
    y : ndarray, shape (3 * n_per_class,)
        Integer labels: 0 = healthy, 1 = minor crack, 2 = major damage.
    """
    np.random.seed(seed)
    blocks = [np.random.multivariate_normal(mean, FEATURE_COV, n_per_class)
              for mean in CLASS_MEANS.values()]
    X = np.vstack(blocks)
    y = np.repeat(np.arange(len(CLASS_MEANS)), n_per_class)
    return X, y


def plot_importance_comparison(rf, gb, X_te, y_te, labels, out_path):
    """Grouped bar chart of impurity-based feature importances for two models.

    Each model's held-out accuracy is shown in the legend. The two importances
    are normalized within each model, so the chart is for comparing *rankings*,
    not raw magnitudes. Saves the figure to ``out_path`` and returns it.
    """
    x = np.arange(len(labels))
    width = 0.35
    rf_acc = np.mean(rf.predict(X_te) == y_te)
    gb_acc = np.mean(gb.predict(X_te) == y_te)

    fig, ax = plt.subplots(figsize=(8.5, 4.5))
    ax.bar(x - width / 2, rf.feature_importances_, width, color='steelblue',
           alpha=0.85, label=f'Random Forest (acc={rf_acc:.2f})')
    ax.bar(x + width / 2, gb.feature_importances_, width, color='darkorange',
           alpha=0.85, label=f'Gradient Boosting (acc={gb_acc:.2f})')
    ax.set_xticks(x)
    ax.set_xticklabels(labels, fontsize=10)
    ax.set_ylabel('Feature importance (impurity-based, normalized within each model)')
    ax.set_title('Feature Importance for Damage State Classification')
    ax.legend()
    ax.grid(True, axis='y', linestyle=':', alpha=0.5)
    fig.tight_layout()
    fig.savefig(out_path, bbox_inches='tight', dpi=150)
    return fig


# --- Build data, fit both ensembles, and report per-class metrics ---------------
X, y = make_damage_dataset()
class_names = list(CLASS_MEANS.keys())
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE, stratify=y, random_state=RANDOM_STATE)

rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS, random_state=RANDOM_STATE).fit(X_tr, y_tr)
gb = GradientBoostingClassifier(
    n_estimators=N_ESTIMATORS, learning_rate=LEARNING_RATE,
    random_state=RANDOM_STATE).fit(X_tr, y_tr)

print("Random Forest:")
print(classification_report(y_te, rf.predict(X_te), target_names=class_names))
print("Gradient Boosting:")
print(classification_report(y_te, gb.predict(X_te), target_names=class_names))

# --- Compare feature-importance rankings (Figure 54.2) --------------------------
fig_path = OUT_DIR / 'fig54_2_feature_importance.png'
plot_importance_comparison(rf, gb, X_te, y_te, FEATURE_LABELS, fig_path)
plt.show()
print(f"Figure saved to {fig_path}")

---
## 2 — A Simulated Bridge: Which Features Predict the Damage State?

This example mirrors the bridge case study in Chapter 54. Twenty strain gages are reduced to one **daily feature vector** over two simulated years. Damage progresses with cumulative use and, when present, *decorrelates* a group of gages near the affected region — a **load-path change**. Each day contributes per-gage peak and RMS strain, a zero-crossing cycle count, a set of **pairwise sensor correlations**, and a running Miner's-rule fatigue accumulator, exactly the reduction shown in the chapter's `day_features` routine (Listing 54.1) that we reproduce below.

The physical hypothesis under test is that *damage redistributes load, and the redistribution shows up first in how sensor pairs move together* — not in any single gage's amplitude. So the correlation features, not the peak strains, should rise to the top of the importance ranking (Figure 54.3).

A note on the split. The chapter describes an ideal *time-ordered* evaluation — train on the first 18 months, test on the final 6 — because a monitoring model must generalize forward in time. Here, though, the object of the analysis is the **feature-importance ranking**, which is computed on the *full* record (as in Listing 54.2). A strict chronological split would place each damage state in its own period and starve the classifier of examples, so we fit the importance model on all the data and keep a stratified holdout only as a quick accuracy sanity check. That very tension — labels running out over time while the model must still generalize forward — is what motivates the unsupervised anomaly detection of Chapter 56.

In [ ]:
# --- Bridge-simulation constants (named, not magic numbers) ---------------------
BRIDGE_SEED         = 7      # seeds the synthetic multichannel strain
N_DAYS              = 730    # ~two years of daily feature vectors
N_GAGE              = 20     # strain gages on the structure
N_SAMP              = 300    # strain samples summarized into each daily vector
NOISE_STD           = 0.15   # per-sample measurement noise (std)
MINER_REF           = 400.0  # reference strain for the crude Miner accumulation
N_ESTIMATORS_BRIDGE = 400    # trees in the bridge random forest
TEST_SIZE_BRIDGE    = 0.3    # stratified holdout fraction (sanity check only)
TOP_K               = 12     # features shown in the importance bar chart

REGION  = np.arange(6, 11)             # gages near the critical region
A_LEVEL = {0: 0.0, 1: 0.35, 2: 0.70}   # shared->independent mixing vs. damage state


def simulate_day(rng, damage_state, sens):
    """Simulate one day of N_GAGE-channel strain (N_SAMP samples).

    All gages share a common daily load history; as ``damage_state`` grows, the
    REGION gages are rotated from that shared signal toward an independent one,
    so their RMS is roughly preserved but their correlation with the rest of the
    array falls. ``rng`` is consumed in a fixed order for reproducibility.
    """
    load = rng.normal(0, 1, N_SAMP)                     # shared daily load history
    shared = np.outer(load, sens)
    G = shared + rng.normal(0, NOISE_STD, (N_SAMP, N_GAGE))
    a = np.clip(A_LEVEL[damage_state] + rng.normal(0, 0.05), 0, 0.95)
    indep = rng.normal(0, 1, (N_SAMP, len(REGION))) * sens[REGION]
    G[:, REGION] = (np.sqrt(1 - a) * shared[:, REGION]
                    + np.sqrt(a) * indep
                    + rng.normal(0, NOISE_STD, (N_SAMP, len(REGION))))
    return G


def day_features(G, pair_sel, miner_prev):
    """Reduce one day of (n_samples x n_gages) strain to a feature vector.

    Returns the concatenated [peak, RMS, cycle-count, selected pairwise
    correlations, cumulative-Miner] vector and the updated Miner accumulator.
    Mirrors Listing 54.1 in the chapter.
    """
    peak = G.max(axis=0)                                # per-gage peak strain
    rms = np.sqrt((G**2).mean(axis=0))                  # per-gage RMS
    # zero-crossings of the mean-removed signal -> a stress-cycle proxy
    cyc = (np.abs(np.diff(np.sign(G - G.mean(0)), axis=0)) > 0).sum(axis=0) / 2
    C = np.corrcoef(G, rowvar=False)                    # full gage-to-gage correlation
    corr = np.array([C[i, j] for i, j in pair_sel])     # selected sensor pairs
    miner = miner_prev + (peak.mean() / MINER_REF)**3   # crude Miner accumulation
    return np.concatenate([peak, rms, cyc, corr, [miner]]), miner


def feature_family(name):
    """Group a feature name into one of four physical families (for coloring)."""
    if name.startswith('corr'):
        return 'corr'
    if name == 'miner':
        return 'miner'
    if name.startswith('cyc'):
        return 'cyc'
    return 'peak/rms'


def plot_bridge_importance(importance, names, out_path, top_k=TOP_K):
    """Horizontal bar chart of the top-``top_k`` random-forest importances,
    colored by feature family. Saves to ``out_path``; returns (fig, order)."""
    order = np.argsort(importance)[::-1][:top_k][::-1]
    cmap = {'corr': '#c8631b', 'miner': '#2e8b57', 'cyc': '#7a8697', 'peak/rms': '#3b6ea5'}
    fig, ax = plt.subplots(figsize=(8, 5))
    ax.barh(range(len(order)), importance[order],
            color=[cmap[feature_family(names[i])] for i in order])
    ax.set_yticks(range(len(order)))
    ax.set_yticklabels([names[i] for i in order])
    ax.set_xlabel('Random-forest feature importance')
    ax.set_title('Bridge Example: Top Predictors of Damage State')
    handles = [plt.Rectangle((0, 0), 1, 1, color=c) for c in cmap.values()]
    ax.legend(handles, cmap.keys(), title='feature family', loc='lower right', fontsize=9)
    fig.tight_layout()
    fig.savefig(out_path, dpi=200, bbox_inches='tight', facecolor='white')
    return fig, order


# --- Generate the two-year monitoring record -----------------------------------
rng = np.random.default_rng(BRIDGE_SEED)
sens = rng.uniform(0.8, 1.2, N_GAGE)                    # per-gage sensitivity

# Cumulative usage drives damage; the state progresses 0 -> 1 -> 2 over two years.
usage = np.cumsum(rng.uniform(0.5, 1.5, N_DAYS))
usage /= usage[-1]
damage = np.where(usage < 0.45, 0, np.where(usage < 0.75, 1, 2))

# Correlation features: region-vs-healthy pairs (informative) + healthy-vs-healthy (control)
pair_sel = [(i, j) for i in REGION for j in range(5)] + [(0, 1), (1, 2), (2, 3), (3, 4)]

feat_rows, miner = [], 0.0
for d in range(N_DAYS):
    G = simulate_day(rng, damage[d], sens)
    feats, miner = day_features(G, pair_sel, miner)
    feat_rows.append(feats)

feature_names = ([f'peak_g{g}' for g in range(N_GAGE)] +
                 [f'rms_g{g}' for g in range(N_GAGE)] +
                 [f'cyc_g{g}' for g in range(N_GAGE)] +
                 [f'corr_{i}_{j}' for i, j in pair_sel] + ['miner'])
X = np.array(feat_rows)
y = damage

# --- Quick sanity check on a stratified holdout, then importances on full data ---
X_tr, X_te, y_tr, y_te = train_test_split(
    X, y, test_size=TEST_SIZE_BRIDGE, stratify=y, random_state=RANDOM_STATE)
rf = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_BRIDGE, random_state=RANDOM_STATE).fit(X_tr, y_tr)
print('stratified holdout accuracy: %.3f' % accuracy_score(y_te, rf.predict(X_te)))

rf_full = RandomForestClassifier(
    n_estimators=N_ESTIMATORS_BRIDGE, random_state=RANDOM_STATE).fit(X, y)
importance = rf_full.feature_importances_

# --- Rank and plot the top predictors (Figure 54.3) -----------------------------
fig_path = OUT_DIR / 'fig54_3_bridge_importance.png'
fig, order = plot_bridge_importance(importance, feature_names, fig_path)
plt.show()
print('most important family:', feature_family(feature_names[order[-1]]))

---
## Where to take this next

- **Cross-check with permutation and SHAP importance.** The bar charts above use *impurity-based* importance, which the chapter warns can favor high-cardinality features and split credit among correlated ones. Recompute the ranking with `sklearn.inspection.permutation_importance` on the held-out set, and with the `shap` library's `TreeExplainer` (TreeSHAP), then see whether the correlation features still dominate. Where the three methods disagree is exactly where the physical story deserves a second look.
- **Aggregate importance by sensor, not by feature.** For the bridge example, sum the importances of all features belonging to each gage — and to each sensor *pair* — to answer the physically meaningful question the chapter poses: *how much does this gage contribute?* rather than ranking individual derived numbers. This grouped view is the one to trust when deciding which channels a lower-cost system can omit.
- **Make the split honestly chronological, then watch it strain.** Replace the stratified holdout with a true time-ordered split (train on days 0–545, test on 546–729) and print the per-class confusion matrix. Because each damage state lives in its own time window, the classifier will struggle — a concrete demonstration of why forward-in-time monitoring pushes you toward the unsupervised anomaly-detection methods of Chapter 56.